# Near-Earth Object Exploratory Data Analysis

This notebook explores asteroid characteristics,
orbital behavior, and close approach patterns using
NASA NeoWs data stored in SQLite.

Author: Anthony Mazza

Project: NASA NEO Data Pipeline

In [9]:
import sqlite3

import pandas as pd
import numpy as np

from pathlib import Path

In [10]:
PROJECT_ROOT = Path.cwd().parent

DB_PATH = PROJECT_ROOT / "data" / "database" / "neows.db"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

conn = sqlite3.connect(DB_PATH)

print(f"Connected to: {DB_PATH}")

Connected to: c:\Users\arcen\git\neo-analytics-platform\data\database\neows.db


In [11]:
asteroids_df = pd.read_sql_query(
    "SELECT * FROM asteroids",
    conn
)

close_approaches_df = pd.read_sql_query(
    "SELECT * FROM close_approaches",
    conn
)

orbital_parameters_df = pd.read_sql_query(
    "SELECT * FROM orbital_parameters",
    conn
)

In [12]:
print("Asteroids:", len(asteroids_df))
print("Close Approaches:", len(close_approaches_df))
print("Orbital Parameters:", len(orbital_parameters_df))

asteroids_df.head()
close_approaches_df.head()
orbital_parameters_df.head()

print(asteroids_df.info())
print(close_approaches_df.info())
print(orbital_parameters_df.info())

Asteroids: 36
Close Approaches: 36
Orbital Parameters: 36
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   asteroid_id                36 non-null     object 
 1   name                       36 non-null     object 
 2   absolute_magnitude_h       36 non-null     float64
 3   estimated_diameter_min_km  36 non-null     float64
 4   estimated_diameter_max_km  36 non-null     float64
 5   is_potentially_hazardous   36 non-null     int64  
dtypes: float64(3), int64(1), object(2)
memory usage: 1.8+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   close_approach_id      36 non-null     int64  
 1   asteroid_id            36 non-null     object 
 2   close_appr

## Missing Value Assessment

We evaluated all major fields in the asteroid,
close approach, and orbital parameter datasets.

For each column we calculated:

- Null count
- Missing percentage

This analysis helps determine whether the dataset
is sufficiently complete for downstream analytics
and visualization.

In [13]:
def missing_value_report(df):
    report = (
        df.isna()
          .sum()
          .to_frame("null_count")
    )

    report["missing_pct"] = (
        report["null_count"] / len(df) * 100
    ).round(2)

    return report.sort_values(
        by="missing_pct",
        ascending=False
    )

In [14]:
for name, df in {
    "Asteroids": asteroids_df,
    "Close Approaches": close_approaches_df,
    "Orbital Parameters": orbital_parameters_df,
}.items():

    print(f"\n{name}")
    print("=" * len(name))

    display(
        missing_value_report(df)
        .query("null_count > 0")
    )


Asteroids


,null_count,missing_pct



Close Approaches


,null_count,missing_pct



Orbital Parameters


,null_count,missing_pct
data_arc_in_days,1,2.78


In [15]:
orbital_parameters_df[
    orbital_parameters_df["data_arc_in_days"].isna()
]

,asteroid_id,orbit_id,orbit_determination_date,first_observation_date,last_observation_date,data_arc_in_days,observations_used,orbit_uncertainty,minimum_orbit_intersection,jupiter_tisserand_invariant,...,perihelion_distance,perihelion_argument,aphelion_distance,perihelion_time,mean_anomaly,mean_motion,equinox,orbit_class_type,orbit_class_description,orbit_class_range
34,3879168,2,2021-04-15 21:55:52,2019-10-10,2019-10-10,NaN,24,6,0.002366,5.968,...,0.652619,109.94359,1.364736,2.461058e+06,304.328748,0.972916,J2000,APO,Near-Earth asteroid orbits which cross the Ear...,a (semi-major axis) > 1.0 AU; q (perihelion) <...


## Missing Value Assessment

The asteroid, close approach, and orbital parameter datasets were evaluated for missing values by calculating null counts and missing percentages for all major fields.

### Findings

#### Asteroids
No missing values were detected in any asteroid fields.

#### Close Approaches
No missing values were detected in any close approach fields.

#### Orbital Parameters
A single missing value was identified in the `data_arc_in_days` field.

| Field | Null Count | Missing % |
|---------|---------|---------|
| data_arc_in_days | 1 | 2.78% |

Further investigation identified the affected record as asteroid `3879168`.

Relevant characteristics of the record include:

- Orbit ID: 2
- First Observation Date: 2019-10-10
- Last Observation Date: 2019-10-10
- Observations Used: 24
- Orbit Uncertainty: 6

Because the first and last observation dates are identical, the missing `data_arc_in_days` value likely originates from the source data rather than from the ingestion or transformation process.

### Conclusion

Data completeness is very high across all datasets. No missing values were found in the asteroid or close approach tables. Only one missing value was detected in the orbital parameters table, affecting 2.78% of records in the current dataset.

The missing value appears to be an isolated source-data issue and is unlikely to materially impact exploratory analysis, visualization, or high-level trend identification. The dataset is considered suitable for further analytical work.

## Data Type Validation

In [16]:
asteroids_df.dtypes

asteroid_id                   object
name                          object
absolute_magnitude_h         float64
estimated_diameter_min_km    float64
estimated_diameter_max_km    float64
is_potentially_hazardous       int64
dtype: object

In [17]:
close_approaches_df.dtypes

close_approach_id          int64
asteroid_id               object
close_approach_date       object
relative_velocity_kps    float64
miss_distance_km         float64
orbiting_body             object
dtype: object

In [18]:
orbital_parameters_df.dtypes

asteroid_id                     object
orbit_id                        object
orbit_determination_date        object
first_observation_date          object
last_observation_date           object
data_arc_in_days               float64
observations_used                int64
orbit_uncertainty               object
minimum_orbit_intersection     float64
jupiter_tisserand_invariant    float64
epoch_osculation               float64
eccentricity                   float64
semi_major_axis                float64
inclination                    float64
ascending_node_longitude       float64
orbital_period                 float64
perihelion_distance            float64
perihelion_argument            float64
aphelion_distance              float64
perihelion_time                float64
mean_anomaly                   float64
mean_motion                    float64
equinox                         object
orbit_class_type                object
orbit_class_description         object
orbit_class_range        

In [19]:
for column in orbital_parameters_df.columns:
    print(column, orbital_parameters_df[column].dtype)

asteroid_id object
orbit_id object
orbit_determination_date object
first_observation_date object
last_observation_date object
data_arc_in_days float64
observations_used int64
orbit_uncertainty object
minimum_orbit_intersection float64
jupiter_tisserand_invariant float64
epoch_osculation float64
eccentricity float64
semi_major_axis float64
inclination float64
ascending_node_longitude float64
orbital_period float64
perihelion_distance float64
perihelion_argument float64
aphelion_distance float64
perihelion_time float64
mean_anomaly float64
mean_motion float64
equinox object
orbit_class_type object
orbit_class_description object
orbit_class_range object
